# Layer 0: Data Collection & Processing

Collects Binance OHLCV and trade tick data via ccxt. Falls back to synthetic data with realistic microstructure when API is unavailable.

## Pipeline Overview

In [ ]:
# Data flow:
# 1. scripts/fetch_binance_data.py  -> data/raw/
# 2. scripts/process_data.py         -> data/processed/
# 3. scripts/validate_data.py        -> validation report

# Scripts use ccxt for Binance API, with synthetic microstructure fallback
# Synthetic data: HMM-like regime transitions, realistic vol/spread per regime

## Raw Data Summary

In [ ]:
import pandas as pd
from pathlib import Path

raw = Path('data/raw')
processed = Path('data/processed')

# Load metadata
import json
with open(raw / 'fetch_metadata.json') as f:
    metadata = json.load(f)
print('Data source:', 'REAL Binance API' if not metadata.get('synthetic_data') else 'SYNTHETIC')
print('Symbols:', metadata.get('symbols'))
print('Generated at:', metadata.get('generated_at'))

## Processed Data Inspection

In [ ]:
price_df = pd.read_parquet(processed / 'price_features.parquet')
trades_df = pd.read_parquet(processed / 'trades_processed.parquet')
print(f'Price features: {price_df.shape}')
print(f'Columns: {list(price_df.columns)}')
print(f'\nTrades: {trades_df.shape}')
print(f'\nDate range: {price_df["timestamp"].min()} to {price_df["timestamp"].max()}')
print(f'\nReturn stats:\n{price_df[["btc_return", "eth_return"]].describe()}')

## Regime Distribution

In [ ]:
import numpy as np

# Use HMM-like classification based on realized_vol for quick inspection
vol = price_df['realized_vol'].dropna()
q33, q66 = vol.quantile([0.33, 0.66])
print(f'Vol quantiles: 33%={q33:.6f}, 66%={q66:.6f}')

regime = pd.cut(vol, bins=[-np.inf, q33, q66, np.inf], labels=['Calm', 'Volatile', 'Stressed'])
print(f'\nQuick regime distribution:\n{regime.value_counts()}')

## Synthetic Data Validation

[Executed if synthetic data was used]